<a href="https://colab.research.google.com/github/ugurcavusoglu/ceng467-alignment-dpo/blob/main/notebooks/ablation_runs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.chdir('/content')
!git clone https://github.com/ugurcavusoglu/ceng467-alignment-dpo.git
os.chdir('/content/ceng467-alignment-dpo')
!pip install transformers==4.44.2 trl==0.10.1 peft==0.12.0 accelerate==0.34.0 datasets bert-score -q


Cloning into 'ceng467-alignment-dpo'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 91 (delta 38), reused 69 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 533.02 KiB | 14.41 MiB/s, done.
Resolving deltas: 100% (38/38), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os

dest = '/content/ceng467-alignment-dpo/models/checkpoints'
src = '/content/drive/MyDrive/ceng467_checkpoints'

if not os.path.exists(dest):
    shutil.copytree(src, dest)
    print("Checkpoints loaded from Drive.")
else:
    print("Checkpoints already exist, skipping.")


Mounted at /content/drive
Checkpoints loaded from Drive.


In [3]:
!pip install evaluate -q


!python scripts/run_eval.py \
  --base_model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --sft_path ./models/checkpoints/sft_50k \
  --dpo_path ./models/checkpoints/dpo_beta02 \
  --n_samples 200


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
2026-05-25 15:24:26.151279: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 15:24:26.224565: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
README.md: 5.77kB [00:00, 25.1MB/s]
harmless-base/train.jsonl.gz: 100% 13.2M/13.2M [00:00<00:00, 15.9MB/s]
helpful-base/train.jsonl.gz: 100% 16.2M/16.2M [00:00<00:00, 39.2MB/s]
helpful-online/train.jsonl.gz: 100% 20.1M/20.1M [00:00<00:00, 49.0MB/s]
helpful-rejection-sampled/train.jsonl.gz: 100% 

In [4]:
!python scripts/run_eval.py \
  --base_model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --sft_path ./models/checkpoints/sft_50k \
  --dpo_path ./models/checkpoints/dpo_beta005 \
  --n_samples 200


2026-05-25 16:15:17.766864: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 16:15:17.840618: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Train samples: 159876
Test samples:  8505
Sample:
{
  "chosen": "I haven't even thought about it.",
  "rejected": "Ass.",
  "prompt": "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here\u2019s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, bu

In [5]:
import json, os

ablation_beta = {
    "zero_shot": {"perplexity": 17.006, "bertscore_f1": 0.8294, "n_samples": 200},
    "sft_50k":   {"perplexity": 9.536,  "bertscore_f1": 0.8171, "n_samples": 200},
    "dpo_beta005": {"perplexity": 18.388, "bertscore_f1": 0.8199, "n_samples": 200},
    "dpo_beta010": {"perplexity": 16.949, "bertscore_f1": 0.8262, "n_samples": 500},
    "dpo_beta020": {"perplexity": 17.653, "bertscore_f1": 0.8291, "n_samples": 200},
}

os.makedirs("results/tables", exist_ok=True)
with open("results/tables/ablation_beta.json", "w") as f:
    json.dump(ablation_beta, f, indent=2)

# Also backup to Drive
with open("/content/drive/MyDrive/ceng467_checkpoints/ablation_beta.json", "w") as f:
    json.dump(ablation_beta, f, indent=2)

print("Saved.")


Saved.


In [7]:
!python scripts/train_dpo.py \
  --subset 10000 \
  --epochs 1 \
  --batch_size 2 \
  --beta 0.1 \
  --lora_rank 8 \
  --ref_model_path ./models/checkpoints/sft_50k \
  --output_dir ./models/checkpoints/dpo_rank8


2026-05-25 17:22:11.328624: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 17:22:11.399871: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Train samples: 10000
Test samples:  8505
Sample:
{
  "chosen": "I haven't even thought about it.",
  "rejected": "Ass.",
  "prompt": "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here\u2019s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, but

In [8]:
!python scripts/train_dpo.py \
  --subset 10000 \
  --epochs 1 \
  --batch_size 2 \
  --beta 0.1 \
  --lora_rank 32 \
  --ref_model_path ./models/checkpoints/sft_50k \
  --output_dir ./models/checkpoints/dpo_rank32


2026-05-25 17:42:43.026850: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 17:42:43.099976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Train samples: 10000
Test samples:  8505
Sample:
{
  "chosen": "I haven't even thought about it.",
  "rejected": "Ass.",
  "prompt": "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here\u2019s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, but

In [9]:
!python scripts/train_dpo.py \
  --subset 10000 \
  --epochs 1 \
  --batch_size 2 \
  --beta 0.1 \
  --lora_rank 16 \
  --lr 5e-5 \
  --ref_model_path ./models/checkpoints/sft_50k \
  --output_dir ./models/checkpoints/dpo_lr5e5


2026-05-25 18:01:03.866049: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 18:01:03.938227: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Train samples: 10000
Test samples:  8505
Sample:
{
  "chosen": "I haven't even thought about it.",
  "rejected": "Ass.",
  "prompt": "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here\u2019s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, but

In [10]:
!python scripts/train_dpo.py \
  --subset 10000 \
  --epochs 1 \
  --batch_size 2 \
  --beta 0.1 \
  --lora_rank 16 \
  --lr 1e-4 \
  --ref_model_path ./models/checkpoints/sft_50k \
  --output_dir ./models/checkpoints/dpo_lr1e4


2026-05-25 18:18:37.619783: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 18:18:37.692216: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Train samples: 10000
Test samples:  8505
Sample:
{
  "chosen": "I haven't even thought about it.",
  "rejected": "Ass.",
  "prompt": "\n\nHuman: What are some cuss words in english?\n\nAssistant: Here\u2019s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, but

In [11]:
import shutil, os

src = '/content/ceng467-alignment-dpo/models/checkpoints'
dst = '/content/drive/MyDrive/ceng467_checkpoints'

shutil.rmtree(dst)
shutil.copytree(src, dst)
print("All checkpoints backed up to Drive.")


All checkpoints backed up to Drive.
